# មេរៀន 08 - លំនាំរចនាភ្នាក់ងារច្រើន


## ការតំឡើង


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ហេតុអ្វីបានជា ប្រព័ន្ធភ្នាក់ងារច្រើន?

ភារកិច្ចក្នុងពិភពពិត ដូចជា ការធ្វើផែនការធ្វើដំណើរ ត្រូវការជំនាញជាច្រើនប្រភេទ — ការរៀបចំដឹកជញ្ជូន និងសម្របសម្រួល, ចំណេះដឹងក្នុងតំបន់, ការគ្រប់គ្រងថវិកា, និងផ្សេងៗទៀត។ ភ្នាក់ងារម្នាក់ដែលព្យាយាមទប់ទល់ជាមួយអ្វីទាំងអស់នោះភ្លាមៗក្លាយទៅជារឿងពិបាកក្នុងការគ្រប់គ្រង។

ប្រព័ន្ធភ្នាក់ងារច្រើនដោះស្រាយបញ្ហានេះតាមរយៈ **ភាពជំនាញពិសេស**: ភ្នាក់ងារនីមួយៗផ្តោតលើវិស័យជាក់លាក់មួយ ដែលបង្កើតលទ្ធផលដែលមានគុណភាពខ្ពស់ជាងអ្នកទូទៅ។ ពួកវាក៏ជួយចម្អែ **សមត្ថភាពពង្រីក** — អ្នកអាចបន្ថែមភ្នាក់ងារថ្មីៗ (ឧ. អ្នកជំនាញអាកាសចរណ៍, អ្នកវិភាគភោជនីយដ្ឋាន) ដោយមិនចាំបាច់សរសេរឡើងវិញលំហូរដំណើរការដែលមានរួច។ ភ្នាក់ងារទាំងនេះផ្សំគ្នាតាមបណ្តូរដំណាក់កាលដែលមានរចនាសម្ព័ន្ធ ហើយបញ្ជូនបរិបទពីមួយទៅមួយ។


## ការបង្កើតភ្នាក់ងារឯកទេស


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## កសាងដំណើរការតម្រៀបជាលំดับ

`WorkflowBuilder` អនុញ្ញាតឲ្យអ្នកភ្ជាប់ភ្នាក់ងារ​ទៅក្នុងក្រាហ្វដែលមានទិសដៅ។ នៅទីនេះ យើងបង្កើតបណ្ដុំដំណើរការ​សាមញ្ញ​ចំនួនពីរជំហាន៖ **TravelPlanner** រៀបចំផែនការធ្វើដំណើរ ហើយ **TravelConcierge** ពិនិត្យ និងបន្ថែមការកែលម្អ។


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ការបន្ថែមភ្នាក់ងារបន្ថែមទៅចរន្តការងារ

មួយក្នុងចំណោមអត្ថប្រយោជន៍ធំបំផុតនៃរចនាប័ទ្មពហុភ្នាក់ងារ គឺភាពងាយស្រួលក្នុងការពង្រីក។ ខាងក្រោមនេះ យើងបានបន្ថែមភ្នាក់ងារ **BudgetReviewer** ដែលពិនិត្យផែនការជាមួយថវិកានៃអ្នកធ្វើដំណើរ សម្គាល់ធាតុដែលអាចធ្វើឱ្យចំណាយលើសកំណត់ និងផ្ដល់ជម្រើសសន្សំប្រាក់។ ចរន្តការងារឥឡូវនេះរត់ភ្នាក់ងារ​បី​នាក់​តាមលំដាប់៖

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## សេចក្តីសង្ខេប

ក្នុងមេរៀននេះ អ្នកបានរៀនពីរបៀប៖

1. **បង្កើតភ្នាក់ងារពិសេស** — នីមួយៗមានតួនាទីផ្ដោត (ការធ្វើផែនការ, ភ្នាក់ងារទទួលភ្ញៀវ, ការពិនិត្យថវិកា).
2. **ភ្ជាប់ភ្នាក់ងារចូលទៅក្នុងដំណើរការតាមលំដាប់** ដោយប្រើ `WorkflowBuilder` និង `add_edge`.
3. **បញ្ជូនលទ្ធផលដោយបន្ត** ពីប្រព័ន្ធដែលមានភ្នាក់ងារច្រើន, តាមដានថាភ្នាក់ងារណាកំពុងនិយាយ។
4. **ពង្រីកដំណើរការ** ដោយបន្ថែមភ្នាក់ងារថ្មីទៅក្នុងខ្សែ ហើយមិនកែប្រែភ្នាក់ងារដែលមានស្រាប់ទេ។

លំនាំរចនាបទភ្នាក់ងារច្រើន បានរក្សាឲ្យភ្នាក់ងារនីមួយៗមានភាពសាមញ្ញ ខណៈដែលបង្កើតលទ្ធផលដែលសម្បូរល្អប្រសើរ និងបានពិនិត្យយ៉ាងទូលំទូលាយ កាន់តែប្រសើរជាងអ្វីដែលភ្នាក់ងារតែមួយអាចសម្រេចបានដោយឯង។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការព្រមាន**:
ឯកសារនេះត្រូវបានបកប្រែដោយប្រើសេវាកម្មបកប្រែ AI [Co-op Translator](https://github.com/Azure/co-op-translator). ទោះបីយើងខិតខំដើម្បីធានាថាមានភាពត្រឹមត្រូវ ក៏ដោយ សូមយល់ដឹងថាការបកប្រែដោយស្វ័យប្រវត្តិអាចមានកំហុស ឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើម​នៅក្នុងភាសាដើម​គួរត្រូវបានចាត់ទុកថាជាប្រភពដែលអាចយោងបាន។ សម្រាប់ព័ត៌មានដែលសំខាន់ យើងសូមណែនាំឱ្យប្រើការបកប្រែដោយមនុស្សវិជ្ជាជីវៈ។ យើងមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសណាមួយដែលកើតឡើងពីការប្រើប្រាស់ការបកប្រែនេះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
